# Afanasiy Nikitin Atlas — a reproducible narrative

This notebook loads the FAIR data spine in [`../data/`](../data/) and **re-derives Khrustalev's 1467–1475 dating anchors from first principles**, then draws the route straight from the data. It is the executable companion to the atlas.

**Run:** `pip install pandas matplotlib`, then `jupyter lab notebooks/atlas.ipynb`.  
**Data:** CC-BY-4.0 (see [`../RIGHTS.md`](../RIGHTS.md)); cite via [`../CITATION.cff`](../CITATION.cff).

In [ ]:
import os
import pandas as pd

DATA = os.path.join('..', 'data')
places   = pd.read_csv(f'{DATA}/places.csv')
itin     = pd.read_csv(f'{DATA}/itinerary.csv').sort_values('seq')
people   = pd.read_csv(f'{DATA}/people.csv')
calendar = pd.read_csv(f'{DATA}/calendar.csv')
events   = pd.read_csv(f'{DATA}/events.csv')

for name, df in [('places', places), ('itinerary', itin), ('people', people),
                 ('calendar', calendar), ('events', events)]:
    print(f'{name:10s} {len(df):>4} rows')

## 1. The dating argument — Orthodox Easter (Julian computus)

Khrustalev's revised chronology rests on the Paschal cycle. We re-derive Orthodox Easter with the classic **Julian computus** (Meeus's algorithm) and check it against the two anchors the journey turns on.

In [ ]:
def julian_easter(year):
    """Orthodox Easter (month, day) in the Julian calendar — Meeus's algorithm."""
    a, b, c = year % 4, year % 7, year % 19
    d = (19 * c + 15) % 30
    e = (2 * a + 4 * b - d + 34) % 7
    month = (d + e + 114) // 31
    day = (d + e + 114) % 31 + 1
    return month, day

for y in range(1467, 1476):
    m, dd = julian_easter(y)
    print(f'{y}: Easter {dd:02d}.{m:02d} (Julian)')

assert julian_easter(1469) == (4, 2),  'Easter 1469 must be 2 April (Julian)'
assert julian_easter(1474) == (4, 10), 'Easter 1474 must be 10 April (Julian)'
print('\nAnchors reproduced: Easter 1469 = 2 Apr (Hormuz), Easter 1474 = 10 Apr (Muscat)  OK')

## 2. Great Lent ∩ Ramadan, spring 1470

The strongest single coincidence: in 1470 Orthodox Great Lent overlapped Ramadan — Afanasiy «postilsya s besermeny». The dates below come from the canonical generator [`../tools/computus.py`](../tools/computus.py) (tabular Islamic calendar), read back from `calendar.csv`.

In [ ]:
c = calendar.set_index('event_id')
lent_1470 = c.loc['lent_1470', 'julian_date']
ram_1470 = calendar[(calendar.type == 'ramadan_start') & (calendar.year_ce == 1470)]['julian_date']
print('Great Lent 1470 begins (Julian):', lent_1470)
print('Ramadan 1470 begins   (Julian):', ram_1470.iloc[0] if len(ram_1470) else 'n/a')
print('\n-> the two fasts overlap in March 1470, reproducing the coincidence Khrustalev builds on')

## 3. The route, drawn from the spine

The 29 waypoints, joined from `itinerary.csv` → `places.csv`. Green = places reconciled to Wikidata.

In [ ]:
import matplotlib.pyplot as plt

pl = places.set_index('place_id')
xs = [pl.loc[r.place_id, 'lon'] for r in itin.itertuples()]
ys = [pl.loc[r.place_id, 'lat'] for r in itin.itertuples()]

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(xs, ys, '-', color='#b2202a', lw=1, alpha=0.6, zorder=1)
colors = ['#2a7a2a' if s == 'confirmed' else '#999' for s in pl['recon_status']]
ax.scatter(pl['lon'], pl['lat'], c=colors, s=30, zorder=3)
for pid, row in pl.iterrows():
    ax.annotate(row['name_ru'].split('(')[0].split('/')[0].strip(), (row['lon'], row['lat']),
                fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_title('Маршрут Афанасия Никитина 1467–1475 (из data/*.csv)')
ax.set_xlabel('lon'); ax.set_ylabel('lat'); ax.grid(alpha=0.2)
plt.show()

conf = (pl['recon_status'] == 'confirmed').sum()
print(f'{len(xs)} waypoints; {conf} places reconciled to Wikidata')

## 4. Provenance & reuse

Every row carries `epistemic` (text / reconstruction / localization / model / hypothesis) and `certainty`; places & people link to Wikidata / GeoNames / Pleiades / VIAF (audit in [`../data/reconciliation.md`](../data/reconciliation.md)). A Linked Places Format gazetteer (`places.lpf.geojson`) and an RDF/Turtle graph (`atlas.ttl`, CIDOC-CRM) are derived from the same spine. Regenerate everything with `python tools/computus.py`, `tools/build_rdf.py`, `tools/build_route.py`, `tools/build_legs.py`; integrity is checked by `tools/validate_data.py` and the `data-validate` CI on every push.